# Earthquakes and Tectonic Plates Analysis

GitHub reconstruction using the **Group 10 Final Project Report (04/17/2025)** as the primary specification. Earlier milestones are used only for non-conflicting implementation details that are absent from the final report.


## 1. Setup

From the repository root, install dependencies and download the two Kaggle datasets:

```bash
pip install -r requirements.txt
python scripts/download_data.py
```


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_processing import load_raw_data, validate_raw_shapes, clean_earthquakes, clean_plates
from src.features import add_distance_to_boundary, model_features
from src.modeling import fit_all_models
from src.visualization import save_eda_plots, save_magnitude_map, save_depth_map

RAW = ROOT / "data" / "raw"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)


## 2. Load the report datasets


In [ ]:
earthquakes_raw, plates_raw = load_raw_data(RAW / "database.csv", RAW / "all.csv")
validate_raw_shapes(earthquakes_raw, plates_raw, ROOT / "data_manifest.json")
print("Earthquakes:", earthquakes_raw.shape)
print("Plate coordinates:", plates_raw.shape)
earthquakes_raw.head()


## 3. Missing-value analysis and cleaning

The project drops columns containing missing values, corrects the three malformed date/time rows, converts Date/Time to numeric time representations, and retains the complete earthquake fields used downstream.


In [ ]:
earthquakes, nulls = clean_earthquakes(earthquakes_raw)
plates = clean_plates(plates_raw)
nulls.sort_values("null_percent", ascending=False)


In [ ]:
earthquakes.info()
earthquakes.head()


## 4. Nearest tectonic-boundary distance

The submitted model-exploration code uses `sklearn.neighbors.KDTree` on `[longitude, latitude]` plate-boundary coordinates and queries the nearest point for each earthquake.


In [ ]:
earthquakes = add_distance_to_boundary(earthquakes, plates)
earthquakes[["Latitude", "Longitude", "Distance To Boundary", "Nearest Plate"]].head()


## 5. Exploratory analysis and visualizations


In [ ]:
save_eda_plots(earthquakes, OUT)
print("Saved EDA figures to", OUT)


## 6. Interactive tectonic maps


In [ ]:
save_magnitude_map(earthquakes, plates, OUT / "earthquake_magnitude_map.html")
save_depth_map(earthquakes, plates, OUT / "earthquake_depth_map.html")
print("Saved interactive maps.")


## 7. Final-report modeling setup

**Inputs:** Latitude, Longitude, Timestamp, Distance To Boundary, Earthquake Type  
**Targets:** Magnitude, Depth  
**Split:** 60% training / 40% validation

Models: Linear Regression, KNN (`k=6`), Random Forest, SVR, and an MLP neural network with one 10-neuron hidden layer, ReLU, Adam, and 1000 maximum iterations.


In [ ]:
X, y_magnitude, y_depth = model_features(earthquakes)
X.head()


In [ ]:
metrics, fitted = fit_all_models(
    X, y_magnitude, y_depth,
    random_state=42,
    test_size=0.40,
    skip_svr=False,
)
metrics


## 8. Final-report validation metrics


In [ ]:
metrics[metrics["split"] == "validation"].reset_index(drop=True)


## 9. Compare reconstruction with the values printed in the final report


In [ ]:
reference = pd.read_csv(ROOT / "docs" / "report_reference_metrics.csv")
validation = metrics[metrics["split"] == "validation"]
comparison = validation.merge(
    reference[reference["split"] == "validation"],
    on=["model", "target", "split"],
    suffixes=("_reconstructed", "_report"),
)
comparison


## Reproducibility note

This repository reconstructs the submitted workflow. The final report does not print every code-level choice (for example, the exact split seed and categorical encoding), so exact metric equality cannot be guaranteed. `docs/REPORT_ALIGNMENT.md` documents which details came from the final report and which non-conflicting details were recovered from earlier submitted milestones.
